# RQ1: Official TSE Format — Zero-Shot vs ICL k=1

**Experiment:** `tse_rq1_official_icl`  
**Setup:** random picking · 3 seeds (0, 3, 6) · ~98 TSE templates  

| Condition | Model | `num_shots` |
|---|---|---|
| **Text · Zero-Shot** | `Qwen/Qwen3-8B-Instruct` | 0 |
| **Text · ICL k=1** | `Qwen/Qwen3-8B-Instruct` | 1 |
| **ChatTS · Zero-Shot** | `bytedance-research/ChatTS-8B` | 0 |
| **ChatTS · ICL k=1** | `bytedance-research/ChatTS-8B` | 1 |
| **VL · Zero-Shot** | `Qwen/Qwen3-VL-8B-Instruct` | 0 |
| **VL · ICL k=1** | `Qwen/Qwen3-VL-8B-Instruct` | 1 |

**Sections:**
1. Load data from W&B
2. Overall accuracy table (all templates pooled)
3. Per-model ICL effect (Δ = ICL − Zero-Shot, bar charts, Wilcoxon test)
4. Per-template comparison heatmap + winner counts
5. Results by TSE difficulty and category
6. Critical Difference diagram (Friedman + Wilcoxon + Holm–Bonferroni)

In [ ]:
import wandb
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import math, networkx
from pathlib import Path
from scipy.stats import wilcoxon, friedmanchisquare
from tqdm.auto import tqdm
import json

if wandb.api.api_key is None:
    for folder in ["", "wandb_logger", "logger"]:
        token_path = Path(folder) / "token.txt"
        if token_path.exists():
            wandb.login(key=token_path.read_text().strip())
            break

ENTITY  = "aviramom-"
PROJECT = "ts-icl"
EXP_ID  = "tse_rq1_official_icl"

TEXT_MODEL   = "Qwen/Qwen3-8B-Instruct"
CHATTS_MODEL = "bytedance-research/ChatTS-8B"
VL_MODEL     = "Qwen/Qwen3-VL-8B-Instruct"
MODELS       = [TEXT_MODEL, CHATTS_MODEL, VL_MODEL]
MODEL_LABELS = {TEXT_MODEL: "Qwen3-8B (Text)", CHATTS_MODEL: "ChatTS-8B", VL_MODEL: "Qwen3-VL-8B"}

# 6 conditions: model × num_shots
CONDITIONS = []
for m in MODELS:
    CONDITIONS += [f"{MODEL_LABELS[m]} · Zero-Shot", f"{MODEL_LABELS[m]} · ICL k=1"]

N_SEEDS = 3  # seeds 0, 3, 6

TSE_META_PATH = Path("../qa_dataset_augmented.json")
if not TSE_META_PATH.exists():
    TSE_META_PATH = Path("qa_dataset_augmented.json")

api = wandb.Api(timeout=60)
print(f"Connected to {ENTITY}/{PROJECT}")
print(f"  Exp ID : {EXP_ID}")

## 1 · Load Data from W&B

Set `FORCE_REFRESH = True` when new runs finish.

In [ ]:
CONFIG_KEYS = ["task_id", "method", "num_samples", "exp_id",
               "num_shots", "random_seed", "picking_strategy", "prompt_format"]
METRIC_KEYS = ["balanced_accuracy", "f1_macro", "f1_weighted",
               "num_of_classes", "total_test_size"]

CACHE_PATH    = Path("cache_tse_rq1_official_icl.parquet")
FORCE_REFRESH = True  # flip to False after first successful fetch

def get_val(cfg, summary, key):
    return summary.get(key, cfg.get(key))

def fetch_exp(exp_id, cache_path):
    if cache_path.exists() and not FORCE_REFRESH:
        df = pd.read_parquet(cache_path)
        print(f"Loaded {len(df)} rows from cache ({cache_path})")
        return df
    all_runs = list(api.runs(path=f"{ENTITY}/{PROJECT}", per_page=1000))
    rows = []
    for run in tqdm(all_runs, desc=f"Processing {exp_id}"):
        summary = {k: v for k, v in (run.summary or {}).items() if not k.startswith("_")}
        cfg     = run.config or {}
        if get_val(cfg, summary, "exp_id") != exp_id:
            continue
        row = {"run_id": run.id, "run_name": run.name, "state": run.state}
        for k in CONFIG_KEYS:
            row[k] = get_val(cfg, summary, k)
        for k in METRIC_KEYS:
            row[k] = get_val(cfg, summary, k)
        rows.append(row)
    df = pd.DataFrame(rows)
    for col in METRIC_KEYS:
        if col in df.columns:
            df[col] = pd.to_numeric(df[col], errors="coerce")
    df.to_parquet(cache_path)
    print(f"Cached {len(df)} rows → {cache_path}")
    return df

df_raw = fetch_exp(EXP_ID, CACHE_PATH)

def assign_condition(row):
    method = str(row.get("method", ""))
    shots  = int(pd.to_numeric(row.get("num_shots", 0), errors="coerce") or 0)
    suffix = "ICL k=1" if shots > 0 else "Zero-Shot"
    for m in MODELS:
        if m in method:
            return f"{MODEL_LABELS[m]} · {suffix}"
    return "Unknown"

df_raw["condition"] = df_raw.apply(assign_condition, axis=1)
df_raw["tid"] = pd.to_numeric(
    df_raw["task_id"].str.replace("icl_tse_", "", regex=False),
    errors="coerce"
).astype("Int64")

print(f"\n{'─'*60}")
print(f"Total runs loaded : {len(df_raw)}")
print(f"Finished          : {(df_raw['state'] == 'finished').sum()}")
print(f"\nCondition counts:")
for cond in CONDITIONS:
    n  = (df_raw["condition"] == cond).sum()
    nf = ((df_raw["condition"] == cond) & (df_raw["state"] == "finished")).sum()
    print(f"  {cond:38s}: {n:4d} total, {nf:4d} finished")

In [ ]:
finished = df_raw[df_raw["state"] == "finished"].copy()
tids     = sorted(finished["tid"].dropna().unique().tolist())

print(f"Templates present in results : {len(tids)}")
print(f"Total finished runs          : {len(finished)}")

# Load TSE metadata
if TSE_META_PATH.exists():
    with open(TSE_META_PATH) as f:
        tse_entries = json.load(f)
    tid_meta = {}
    for e in tse_entries:
        t = e["tid"]
        if t not in tid_meta:
            tid_meta[t] = {
                "difficulty":  e.get("difficulty",  "unknown"),
                "category":    e.get("category",    "unknown"),
                "subcategory": e.get("subcategory", "unknown"),
                "question":    e.get("question",    ""),
            }
    print(f"Loaded metadata for {len(tid_meta)} tids")
else:
    tid_meta = {}
    print("WARNING: qa_dataset_augmented.json not found")

finished["difficulty"]  = finished["tid"].map(lambda t: tid_meta.get(t, {}).get("difficulty", "unknown"))
finished["category"]    = finished["tid"].map(lambda t: tid_meta.get(t, {}).get("category",   "unknown"))
finished["question"]    = finished["tid"].map(lambda t: tid_meta.get(t, {}).get("question",   ""))

# Coverage check
count_check = finished.pivot_table(
    index="tid", columns="condition",
    values="balanced_accuracy", aggfunc="count"
).fillna(0).astype(int)
incomplete = count_check[(count_check < N_SEEDS).any(axis=1)]
print(f"\nTemplates with < {N_SEEDS} seeds in any condition: {len(incomplete)}")
if len(incomplete):
    display(incomplete)

## 2 · Overall Accuracy — All Templates Pooled

- **Macro avg** — unweighted mean of per-template balanced accuracy
- **Weighted avg** — weighted by `total_test_size`

In [ ]:
overall_rows = []
for cond in CONDITIONS:
    cdf = finished[finished["condition"] == cond]
    if cdf.empty:
        continue
    per_tid = cdf.groupby("tid").agg(
        bal_acc=("balanced_accuracy", "mean"),
        test_size=("total_test_size", "mean"),
    ).dropna(subset=["bal_acc"])
    macro   = per_tid["bal_acc"].mean()
    weights = per_tid["test_size"].fillna(1)
    weighted = np.average(per_tid["bal_acc"], weights=weights)
    overall_rows.append({
        "Condition":           cond,
        "Templates":           len(per_tid),
        "Macro Avg BalAcc":    macro,
        "Weighted Avg BalAcc": weighted,
    })

overall_df = pd.DataFrame(overall_rows).set_index("Condition")
display(
    overall_df.style
    .format({"Templates": "{:.0f}", "Macro Avg BalAcc": "{:.4f}", "Weighted Avg BalAcc": "{:.4f}"})
    .background_gradient(cmap="RdYlGn", axis=0,
                         subset=["Macro Avg BalAcc", "Weighted Avg BalAcc"], vmin=0, vmax=1)
    .set_caption("Overall benchmark accuracy — 6 conditions · 98 TSE templates · 3 seeds")
)

# Grouped bar chart: zero-shot vs ICL per model
model_short   = [MODEL_LABELS[m] for m in MODELS]
zs_vals  = [overall_df.loc[f"{lbl} · Zero-Shot", "Macro Avg BalAcc"]
             if f"{lbl} · Zero-Shot" in overall_df.index else np.nan for lbl in model_short]
icl_vals = [overall_df.loc[f"{lbl} · ICL k=1", "Macro Avg BalAcc"]
             if f"{lbl} · ICL k=1" in overall_df.index else np.nan for lbl in model_short]

x    = np.arange(len(model_short))
w    = 0.35
fig, ax = plt.subplots(figsize=(9, 5))
bars_zs  = ax.bar(x - w/2, zs_vals,  width=w, label="Zero-Shot", color="#1565C0", edgecolor="white")
bars_icl = ax.bar(x + w/2, icl_vals, width=w, label="ICL k=1",   color="#4CAF50", edgecolor="white")
for bars in [bars_zs, bars_icl]:
    for bar in bars:
        h = bar.get_height()
        if not np.isnan(h):
            ax.text(bar.get_x() + bar.get_width()/2, h + 0.005, f"{h:.3f}",
                    ha="center", va="bottom", fontsize=9, fontweight="bold")
ax.set_xticks(x); ax.set_xticklabels(model_short, fontsize=10)
ax.set_ylabel("Macro Balanced Accuracy")
ax.set_title("RQ1: Zero-Shot vs ICL k=1 — 3 Models · ~98 TSE Templates")
ax.legend(fontsize=10)
ax.set_ylim(0, max(max(zs_vals + [0]), max(icl_vals + [0])) * 1.2)
plt.tight_layout()
plt.show()

## 3 · Per-Model ICL Effect

Δ = ICL k=1 − Zero-Shot, paired by template. Green = ICL helps.

In [ ]:
mean_p = finished.pivot_table(
    index="tid", columns="condition",
    values="balanced_accuracy", aggfunc="mean"
)
mean_p.columns.name = ""

# Add metadata
mean_p["Difficulty"] = mean_p.index.map(lambda t: tid_meta.get(t, {}).get("difficulty", "?"))
mean_p["Category"]   = mean_p.index.map(lambda t: tid_meta.get(t, {}).get("category",   "?"))
mean_p["Question"]   = mean_p.index.map(lambda t: tid_meta.get(t, {}).get("question",   "")[:55])

def sig(p):
    return "*** (p<0.001)" if p < 0.001 else "** (p<0.01)" if p < 0.01 else "* (p<0.05)" if p < 0.05 else "n.s."

model_triples = [
    (f"{MODEL_LABELS[m]} · Zero-Shot", f"{MODEL_LABELS[m]} · ICL k=1", MODEL_LABELS[m])
    for m in MODELS
    if f"{MODEL_LABELS[m]} · Zero-Shot" in mean_p.columns and f"{MODEL_LABELS[m]} · ICL k=1" in mean_p.columns
]

fig, axes = plt.subplots(1, len(model_triples),
                         figsize=(10 * len(model_triples), max(5, len(tids) * 0.22)),
                         sharey=True)
if len(model_triples) == 1:
    axes = [axes]

for ax, (cond_zs, cond_icl, label) in zip(axes, model_triples):
    delta = (mean_p[cond_icl] - mean_p[cond_zs]).dropna().sort_values(ascending=False)
    colors = ["#4CAF50" if v >= 0 else "#F44336" for v in delta]
    ax.barh(delta.index.astype(str)[::-1], delta.values[::-1], color=colors[::-1], height=0.75)
    ax.axvline(0, color="black", linewidth=0.8)
    mean_d = delta.mean()
    ax.axvline(mean_d, color="navy", linewidth=1.2, linestyle="--", label=f"Mean Δ = {mean_d:+.3f}")
    ax.set_xlabel("Δ Balanced Accuracy (ICL k=1 − Zero-Shot)")
    ax.set_ylabel("Template ID (tid)")
    ax.set_title(label)
    ax.legend(fontsize=9)

    print(f"{label}:")
    print(f"  Templates           : {len(delta)}")
    print(f"  Mean (Zero-Shot)    : {mean_p[cond_zs].dropna().mean():.4f}")
    print(f"  Mean (ICL k=1)      : {mean_p[cond_icl].dropna().mean():.4f}")
    print(f"  Mean Δ              : {mean_d:+.4f}")
    print(f"  Templates improved  : {(delta > 0).sum()} / {len(delta)}  ({(delta > 0).mean()*100:.1f}%)")
    print(f"  Templates hurt      : {(delta < 0).sum()} / {len(delta)}  ({(delta < 0).mean()*100:.1f}%)")
    if len(delta.dropna()) >= 2:
        stat, p = wilcoxon(mean_p[cond_icl].dropna().reindex(delta.index),
                           mean_p[cond_zs].dropna().reindex(delta.index),
                           zero_method="pratt", alternative="two-sided", nan_policy="omit")
        print(f"  Wilcoxon (two-sided): p={p:.4f}  {sig(p)}")
    print()

plt.suptitle("Δ Balanced Accuracy per Template (ICL k=1 − Zero-Shot)", y=1.01, fontsize=13)
plt.tight_layout()
plt.show()

### Wilcoxon Signed-Rank Tests — ICL vs Zero-Shot per Model

In [ ]:
wilcox_rows = []
for cond_zs, cond_icl, label in model_triples:
    paired = mean_p[[cond_zs, cond_icl]].dropna()
    if len(paired) < 2:
        print(f"{label}: insufficient paired data"); continue
    zs_v, icl_v = paired[cond_zs].values, paired[cond_icl].values
    delta_v = icl_v - zs_v

    stat_2, p_2 = wilcoxon(icl_v, zs_v, zero_method="pratt", alternative="two-sided")
    stat_g, p_g = wilcoxon(icl_v, zs_v, zero_method="pratt", alternative="greater")
    stat_l, p_l = wilcoxon(icl_v, zs_v, zero_method="pratt", alternative="less")

    print(f"{'─'*60}")
    print(f"{label}")
    print(f"  Paired templates    : {len(paired)}")
    print(f"  Mean (Zero-Shot)    : {zs_v.mean():.4f}")
    print(f"  Mean (ICL k=1)      : {icl_v.mean():.4f}")
    print(f"  Mean Δ              : {delta_v.mean():+.4f}")
    print(f"  Templates improved  : {(delta_v > 0).sum()} / {len(delta_v)}  ({(delta_v > 0).mean()*100:.1f}%)")
    print(f"  Templates hurt      : {(delta_v < 0).sum()} / {len(delta_v)}  ({(delta_v < 0).mean()*100:.1f}%)")
    print(f"  Wilcoxon two-sided  : stat={stat_2:.1f},  p={p_2:.4f}  {sig(p_2)}")
    print(f"  Wilcoxon (ICL>ZS)   : stat={stat_g:.1f},  p={p_g:.4f}  {sig(p_g)}")
    print()
    wilcox_rows.append({"Model": label, "N": len(paired),
                        "ZeroShot": zs_v.mean(), "ICL_k1": icl_v.mean(),
                        "Δ": delta_v.mean(), "p (two-sided)": p_2, "p (ICL>ZS)": p_g,
                        "Sig.": sig(p_2)})

wilcox_df = pd.DataFrame(wilcox_rows).set_index("Model")
display(
    wilcox_df.style
    .format({"ZeroShot": "{:.4f}", "ICL_k1": "{:.4f}", "Δ": "{:+.4f}",
             "p (two-sided)": "{:.4f}", "p (ICL>ZS)": "{:.4f}"})
    .background_gradient(cmap="RdBu", subset=["Δ"], axis=0)
    .set_caption("Wilcoxon signed-rank: ICL k=1 vs Zero-Shot per model")
)

## 4 · Per-Template Comparison Heatmap + Winner Counts

In [ ]:
cond_cols = [c for c in CONDITIONS if c in mean_p.columns]
tbl = mean_p[cond_cols].reindex(tids)

display(
    tbl.style
    .background_gradient(cmap="RdYlGn", axis=None, vmin=0, vmax=1)
    .format("{:.3f}", na_rep="—")
    .set_caption("Balanced Accuracy — all 6 conditions (mean over 3 seeds)")
)

# Delta columns: ICL - ZeroShot per model
for cond_zs, cond_icl, label in model_triples:
    tbl[f"Δ {label}"] = tbl[cond_icl] - tbl[cond_zs]

delta_cols = [f"Δ {t[2]}" for t in model_triples if f"Δ {t[2]}" in tbl.columns]
if delta_cols:
    vabs = tbl[delta_cols].abs().max().max()
    display(
        tbl[delta_cols].style
        .background_gradient(cmap="RdBu", axis=None, vmin=-vabs, vmax=vabs)
        .format("{:+.3f}", na_rep="—")
        .set_caption("Δ per model (ICL k=1 − Zero-Shot)")
    )

# Win counts
winners  = tbl[cond_cols].idxmax(axis=1)
win_cnts = winners.value_counts().reindex(cond_cols, fill_value=0)
total    = len(tbl.dropna(how="all"))
print(f"\nWin counts (out of {total} templates):")
summary = pd.DataFrame({"Templates Won": win_cnts, "Win %": (win_cnts / total * 100).round(1)})
display(
    summary.style
    .background_gradient(cmap="YlGn", subset=["Templates Won"], axis=0)
    .format({"Templates Won": "{:d}", "Win %": "{:.1f}%"})
)

## 5 · Results by TSE Difficulty and Category

In [ ]:
for group_col, group_label in [("difficulty", "Difficulty"), ("category", "Category")]:
    rows = []
    for grp in sorted(finished[group_col].dropna().unique()):
        grp_df = finished[finished[group_col] == grp]
        row = {group_label: grp, "N Templates": grp_df["tid"].nunique()}
        for cond in cond_cols:
            cdf = grp_df[grp_df["condition"] == cond]
            if cdf.empty:
                row[cond] = np.nan
                continue
            per_tid = cdf.groupby("tid").agg(
                bal_acc=("balanced_accuracy", "mean"),
                test_size=("total_test_size", "mean")
            ).dropna(subset=["bal_acc"])
            row[cond] = np.average(per_tid["bal_acc"], weights=per_tid["test_size"].fillna(1))
        rows.append(row)
    grp_df_out = pd.DataFrame(rows).set_index(group_label)
    display(
        grp_df_out.style
        .format({"N Templates": "{:.0f}", **{c: "{:.4f}" for c in cond_cols if c in grp_df_out.columns}})
        .background_gradient(cmap="RdYlGn", axis=None,
                             subset=[c for c in cond_cols if c in grp_df_out.columns], vmin=0, vmax=1)
        .set_caption(f"Weighted Avg Accuracy by {group_label}")
    )

## 6 · Critical Difference Diagram

All 6 conditions ranked via **Friedman test** + **pairwise Wilcoxon + Holm–Bonferroni** at α=0.05.

In [ ]:
pivot_stat = mean_p[cond_cols].dropna()
print(f"Templates used for CD diagram: {len(pivot_stat)}")

groups = [pivot_stat[c].values for c in cond_cols]
if len(groups) >= 2 and all(len(g) == len(groups[0]) for g in groups):
    friedman_stat, friedman_p = friedmanchisquare(*groups)
    print(f"Friedman test: χ²={friedman_stat:.4f},  p={friedman_p:.6f}  {sig(friedman_p)}")

    # Pairwise Wilcoxon with Holm-Bonferroni
    p_values = []
    for i in range(len(cond_cols) - 1):
        for j in range(i + 1, len(cond_cols)):
            c1, c2 = cond_cols[i], cond_cols[j]
            d1, d2 = pivot_stat[c1].values, pivot_stat[c2].values
            try:
                p = wilcoxon(d1, d2, zero_method="pratt")[1]
            except ValueError:
                p = 1.0
            p_values.append([c1, c2, p, False])

    p_values.sort(key=lambda x: x[2])
    m_pairs = len(p_values)
    for i, pv in enumerate(p_values):
        pv[3] = pv[2] <= 0.05 / (m_pairs - i)

    print("\nPairwise Wilcoxon + Holm–Bonferroni:")
    for pv in p_values:
        marker = "✓ sig." if pv[3] else "✗ n.s."
        print(f"  [{marker}]  {pv[0]}  vs  {pv[1]}  — p={pv[2]:.4f}")

    avg_ranks = pivot_stat.rank(axis=1, ascending=False).mean(axis=0).sort_values()
    print("\nAverage ranks (1 = best):")
    for name, rank in avg_ranks.items():
        print(f"  {name:40s}: {rank:.4f}")
else:
    print("Insufficient data for CD diagram")

## 7 · Summary

In [ ]:
print("=" * 65)
print(f"Experiment  : {EXP_ID}")
print(f"Templates   : {len(tids)} / 98")
print(f"Seeds       : {N_SEEDS}  (0, 3, 6)")
print("=" * 65)

print("\nOverall Macro Balanced Accuracy:")
for cond in cond_cols:
    cdf = finished[finished["condition"] == cond]
    per_tid = cdf.groupby("tid")["balanced_accuracy"].mean().dropna()
    print(f"  {cond:42s}: {per_tid.mean():.4f}")

print("\nICL Effect Summary (Δ = ICL k=1 − Zero-Shot):")
for cond_zs, cond_icl, label in model_triples:
    paired = mean_p[[cond_zs, cond_icl]].dropna()
    if paired.empty:
        continue
    delta = paired[cond_icl] - paired[cond_zs]
    stat, p = wilcoxon(paired[cond_icl].values, paired[cond_zs].values,
                       zero_method="pratt", alternative="two-sided") if len(paired) >= 2 else (np.nan, np.nan)
    print(f"\n  {label}:")
    print(f"    Mean Δ              : {delta.mean():+.4f}")
    print(f"    Templates improved  : {(delta > 0).sum()} / {len(delta)}  ({(delta > 0).mean()*100:.1f}%)")
    print(f"    Templates hurt      : {(delta < 0).sum()} / {len(delta)}  ({(delta < 0).mean()*100:.1f}%)")
    print(f"    Wilcoxon two-sided  : p={p:.4f}  {sig(p)}")